In [ ]:
class SystemPromptBase:
    def __init__(
        self,
        use_example: bool=False,
        prefix: str="SYSTEM IDENTITY \
                    You are OpenSI-CoSMIC, a helpful assistant developed by Open Source Institute at University of Canberra. \
                         You would have access to conversation history, this is for your context only. \
                             Allways be polite and provide formal responses. \
                                 If you don't know the answer, say you don't know, but try to provide some helpful information if possible. \
                                     Always follow the format of your response as: \
                                         1. Answer: your answer here. \
                                         2. Explanation: your explanation here. \
                                            3. Reference: your reference here if applicable. \
    ):
        """System prompt base.

        Args:
            use_example (bool, optional): use example in system prompt to detect keywords for
                response truncation. Defaults to False.
            prefix (str): prefix to start the prompt. Default to "".
        """
        self.use_example = use_example
        self.prefix = prefix

    def set_prefix(
        self,
        prefix: str
    ):
        """Set prefix externally.

        Args:
            prefix (str): prefix for system prompt.
        """
        self.prefix = prefix

    def get_context(
        self,
        context: str=""
    ):
        """Get context based on the input type.

        Args:
            context (str|dict, optional): context, string or dictionary.
                Defaults to "".

        Returns:
            context: extract context or an empty string.
        """
        if isinstance(context, dict):
            if "context" in context:
                context = context["context"]
            else:
                context = ""

        return context

    def set_use_example(
        self,
        use_example: bool
    ):
        """Set use_example externally.

        Args:
            use_example (bool): use example in system prompt.
        """
        self.use_example = use_example

    def __call__(
        self,
        user_prompt: str,
        context: str=""
    ):
        """Merge user_prompt in system prompt as the question containing context.

        Args:
            user_prompt (str): user prompt.
            context (str|dict, optional): context retrieved if applicable. Defaults to "".
        """
        # Need to be implemented, otherwise raise error.
        raise NotImplementedError

SyntaxError: unterminated string literal (detected at line 16) (7680737.py, line 5)

In [ ]:
from pathlib import Path

def load_service_prompt(services: str) -> str:
    """
    Attempt to load additional sytem prompt content from a text file under:
        ./src/services/llms/prompts/<services> or <services>.txt

    If file is not found or `services` is falsy, return empty string.
    """
    if not services or not isinstance(services, str):
        return ""

    prompts_root = Path.cwd() / "src" / "services" / "llms" / "prompts"
    # Try exact filename first (no extension), then .txt
    candidate_paths = [
        prompts_root / services,                 # e.g., prompts/promptA
        prompts_root / f"{services}.txt",        # e.g., prompts/promptA.txt
    ]

    for p in candidate_paths:
        try:
            if p.is_file():
                return p.read_text(encoding="utf-8")
        except Exception:
            # Swallow read errors silently and fall through to default behavior.
            # (You can log here if you have a logger, e.g., logger.warning(...))
            pass

    # If no file found/readable
    return ""


In [9]:
load_service_prompt("AcademicGovernance")

'PRIMARY MISSION (STRICT SCOPE) \\\n- Answer only questions related to Academic Governance and Research Integrity at the University of Canberra (UC). \\\n- Relevant areas include: academic integrity, research integrity, ethics (human/animal), authorship and contributor roles, supervision, HDR governance, assessment and appeals, course and award approvals, academic board/committees, policy interpretation, escalation pathways, and compliance/reporting frameworks at UC. \\\n- Do not answer content unrelated to UC academic governance or research integrity (e.g., unrelated tech support, other universities, medical/financial/legal advice, general programming, personal matters). \\\nBEHAVIOUR RULES \\\n1) Accuracy & Conciseness: Provide precise, succinct responses. If policy nuance matters, list key clauses or official UC policy titles (without fabricating). \\\n2) Uncertainty Handling: If you are not sure, state the uncertainty and suggest appropriate UC contacts/resources (e.g., Academic Go

In [11]:
class GPT(SystemPromptBase):
    def __init__(self, **kwargs):
        """For GPT API.
        """
        super().__init__(**kwargs)
        self._prompts_root = Path.cwd() / "src" / "services" / "llms" / "prompts"
        
    def _load_service_prompt(self,services: str) -> str:
        """
        Attempt to load additional sytem prompt content from a text file under:
            ./src/services/llms/prompts/<services> or <services>.txt

        If file is not found or `services` is falsy, return empty string.
        """
        if not services or not isinstance(services, str):
            return ""

        prompts_root = self._prompts_root
        # Try exact filename first (no extension), then .txt
        candidate_paths = [
            prompts_root / services,                 # e.g., prompts/promptA
            prompts_root / f"{services}.txt",        # e.g., prompts/promptA.txt
        ]

        for p in candidate_paths:
            try:
                if p.is_file():
                    return p.read_text(encoding="utf-8")
            except Exception:
                pass

        # If no file found/readable
        return ""

    def __call__(
        self,
        user_prompt: str,
        context: str="",
        services: str=""
    ):
        """Apply system prompt with user prompt and context.

        Args:
            user_prompt (str): question with context.
            context (str|dict, optional): context retrieved. Defaults to "".
            services (str, optional): service name for loading specific system prompt. Defaults to "".

        Returns:
            system_prompt (str): system prompt with question and context under LLM query format.
        """
    
        
        # Compose the system content: base self.prefix + (optional) file content
        prompt_service = self._load_service_prefix(services)
        composed_prefix = self.prefix + (prompt_service if prompt_service else "")

        
        system_prompt = [
            {
                "role": "system",          
                "content": composed_prefix
            },
            {"role": "user", "content": user_prompt}
        ]

        return system_prompt

NameError: name 'SystemPromptBase' is not defined

In [10]:
test=GPT()

NameError: name 'GPT' is not defined

In [8]:
test.__call__("What is the process for reporting suspected research misconduct at UC?")

[{'role': 'system',
  'content': 'SYSTEM IDENTITY                     You are OpenSI-CoSMIC, a helpful assistant developed by Open Source Institute at University of Canberra.                          You would have access to conversation history, this is for your context only.                     PRIMARY MISSION (STRICT SCOPE)                     - Answer only questions related to Academic Governance and Research Integrity at the University of Canberra (UC).                     - Relevant areas include: academic integrity, research integrity, ethics (human/animal), authorship and contributor roles, supervision, HDR governance, assessment and appeals, course and award approvals, academic board/committees, policy interpretation, escalation pathways, and compliance/reporting frameworks at UC.                     - Do not answer content unrelated to UC academic governance or research integrity (e.g., unrelated tech support, other universities, medical/financial/legal advice, general progra